# 03 — Extraction test

A thin frontend to `src/extract.py` for **visually inspecting** the structured output of a
single live Gemini call, before wiring up the full `data/in/interviews.json` loop.

> **This notebook always makes a real Gemini API call** when you run the cells below.
> `RUN_LIVE_TESTS` does **not** apply here — that flag only gates the live *pytest* in
> `tests/test_extract.py`, never `extract()`. A `503 … high demand` error just means Gemini was
> temporarily busy; rerun shortly.

Run one transcript through `extract()` and print the resulting `PatientLabels` as indented
JSON, so every field (enums, demographics, `referral_pathway`, …) can be checked by eye.

`extract()` loads `GEMINI_API_KEY` from `.env` itself — no extra setup needed.

In [1]:
import json
import logging
import sys

import litellm

sys.path.append("../src")  # kernel cwd is notebooks/ (per %pwd); src/ is one level up

from extract import LOG_DIR, SYSTEM_PROMPT, TESTS_OUT, extract
from schema import PatientLabels

litellm._turn_on_debug()  # verbose LiteLLM logging for debugging API calls (e.g. 503s)

# litellm logs via the stdlib `logging` module (not loguru); attach a file handler to the
# "LiteLLM" logger so the verbose debug stream persists to logs/litellm_debug.log.
logging.getLogger("LiteLLM").addHandler(
    logging.FileHandler(LOG_DIR / "litellm_debug.log")
)

In [2]:
# Synthetic transcript with unambiguous facts (same as tests/test_extract.py).
transcript = (
    "I'm Alex, 38, male. Diagnosed with Crohn's four years ago. After mesalamine and "
    "prednisone failed, my gastroenterologist started me on Humira, which I've taken ever "
    "since and it keeps me in remission."
)

## What gets sent to Gemini

Before the call returns, inspect what the model is conditioned on — mirrors the arguments
`extract()` builds for `litellm.completion()`: the **system prompt** (`data/prompts/system.txt`)
and the **Pydantic schema** passed as `response_schema` (how the JSON output is constrained).

In [3]:
# Same shape extract() sends (see src/extract.py) — shown here to inspect it before the call.
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f"patient_id: P000\n\n{transcript}"},
]
response_format = {
    "type": "json_object",
    "response_schema": PatientLabels.model_json_schema(),
    "enforce_validation": True,
}

print("=== SYSTEM PROMPT (data/prompts/system.txt) ===")
print(messages[0]["content"])
print("\n=== USER MESSAGE ===")
print(messages[1]["content"])
print(
    "\n=== response_format.response_schema (PatientLabels JSON schema sent to Gemini) ==="
)
print(json.dumps(response_format["response_schema"], indent=2))

=== SYSTEM PROMPT (data/prompts/system.txt) ===
Extract the structured labels from this Crohn's disease patient interview. Return only valid JSON matching the schema — no prose, no code fences.

Use the schema's enum values exactly. Leave a field null or empty when the transcript does not support a confident value. Set patient_id to the value given in the message.

churn: set true when the interview looks truncated or the patient disengaged before their story resolved — the narrative is cut off mid-journey, trails off, is very short or vague, or ends with no clear outcome. Set false when the interview reads as a reasonably complete journey. Keep this truncation signal distinct from a topic simply being absent from an otherwise complete story (that is "not mentioned", not churn).

treatment_records: for each entry, set before_biologic true if the treatment was tried before the patient started or was offered a biologic, and false otherwise.

referral_pathway: list the patient's journey a

In [4]:
# out_dir=TESTS_OUT keeps this synthetic prediction in data/out/tests, not production.
labels = extract(transcript, "P000", out_dir=TESTS_OUT)
print(labels.model_dump_json(indent=2))

09:10:29 - LiteLLM:DEBUG: utils.py:485 - 

09:10:29 - LiteLLM:DEBUG: utils.py:485 - Request to litellm:
09:10:29 - LiteLLM:DEBUG: utils.py:485 - litellm.completion(model='gemini/gemini-2.5-flash-lite', num_retries=2, drop_params=True, messages=[{'role': 'system', 'content': 'Extract the structured labels from this Crohn\'s disease patient interview. Return only valid JSON matching the schema — no prose, no code fences.\n\nUse the schema\'s enum values exactly. Leave a field null or empty when the transcript does not support a confident value. Set patient_id to the value given in the message.\n\nchurn: set true when the interview looks truncated or the patient disengaged before their story resolved — the narrative is cut off mid-journey, trails off, is very short or vague, or ends with no clear outcome. Set false when the interview reads as a reasonably complete journey. Keep this truncation signal distinct from a topic simply being absent from an otherwise complete story (that is "not 

{
  "patient_id": "P000",
  "churn": false,
  "incomplete_journey": false,
  "demographics": {
    "gender": "male",
    "age": 38
  },
  "biologic_prescribed": true,
  "biologic_taken": true,
  "biologic_not_mentioned": false,
  "biologic_type": "Humira",
  "reasons_for_biologic_prescribed": "DOCTOR_CHOICE",
  "reasons_for_biologic_not_taken": "NOT_APPLICABLE",
  "comorbid_conditions": [],
  "treatment_records": [
    {
      "name": "mesalamine",
      "treatment_class": "conventional",
      "outcome": "FAILED",
      "reason_stopped": null,
      "before_biologic": true
    },
    {
      "name": "prednisone",
      "treatment_class": "conventional",
      "outcome": "FAILED",
      "reason_stopped": null,
      "before_biologic": true
    },
    {
      "name": "Humira",
      "treatment_class": "biologic",
      "outcome": "SUCCESS",
      "reason_stopped": null,
      "before_biologic": false
    }
  ],
  "treatment_outcome": "SUCCESS",
  "referral_pathway": [
    "symptom_onset